In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time
np.random.seed(42)

In [2]:
class ChargedParticlesSim(object):
    def __init__(self, n_balls=5, box_size=5., loc_std=1., vel_norm=0.5,
                 interaction_strength=1., noise_var=0.):
        self.n_balls = n_balls
        self.box_size = box_size
        self.loc_std = loc_std
        self.loc_std = loc_std * (float(n_balls)/5.) ** (1/3)
        print(self.loc_std)
        self.vel_norm = vel_norm
        self.interaction_strength = interaction_strength
        self.noise_var = noise_var


        self._charge_types = np.array([-1., 0., 1.])
        self._delta_T = 0.001
        self._max_F = 0.1 / self._delta_T
        self.dim = 3

    def _l2(self, A, B):
        """
        Input: A is a Nxd matrix
               B is a Mxd matirx
        Output: dist is a NxM matrix where dist[i,j] is the square norm
            between A[i,:] and B[j,:]
        i.e. dist[i,j] = ||A[i,:]-B[j,:]||^2
        """
        A_norm = (A ** 2).sum(axis=1).reshape(A.shape[0], 1)
        B_norm = (B ** 2).sum(axis=1).reshape(1, B.shape[0])
        dist = A_norm + B_norm - 2 * A.dot(B.transpose())
        return dist

    def _energy(self, loc, vel, edges):

        # disables division by zero warning, since I fix it with fill_diagonal
        with np.errstate(divide='ignore'):

            K = 0.5 * (vel ** 2).sum()
            U = 0
            for i in range(loc.shape[1]):
                for j in range(loc.shape[1]):
                    if i != j:
                        r = loc[:, i] - loc[:, j]
                        dist = np.sqrt((r ** 2).sum())
                        U += 0.5 * self.interaction_strength * edges[
                            i, j] / dist
            return U + K

    def _clamp(self, loc, vel):
        '''
        :param loc: 2xN location at one time stamp
        :param vel: 2xN velocity at one time stamp
        :return: location and velocity after hiting walls and returning after
            elastically colliding with walls
        '''
        assert (np.all(loc < self.box_size * 3))
        assert (np.all(loc > -self.box_size * 3))

        over = loc > self.box_size
        loc[over] = 2 * self.box_size - loc[over]
        assert (np.all(loc <= self.box_size))

        # assert(np.all(vel[over]>0))
        vel[over] = -np.abs(vel[over])

        under = loc < -self.box_size
        loc[under] = -2 * self.box_size - loc[under]
        # assert (np.all(vel[under] < 0))
        assert (np.all(loc >= -self.box_size))
        vel[under] = np.abs(vel[under])

        return loc, vel

    def sample_trajectory(self,charges,init_loc,init_vel, T=10000, sample_freq=10):
        n = self.n_balls
        assert (T % sample_freq == 0)
        T_save = int(T / sample_freq - 1)
        diag_mask = np.ones((n, n), dtype=bool)
        np.fill_diagonal(diag_mask, 0)
        counter = 0
        # Sample edges
        charges = charges
        edges = charges.dot(charges.transpose())
        # Initialize location and velocity
        loc = np.zeros((T_save, self.dim, n))
        vel = np.zeros((T_save, self.dim, n))
        loc_next = init_loc
        vel_next = init_vel
        v_norm = np.sqrt((vel_next ** 2).sum(axis=0)).reshape(1, -1)
        vel_next = vel_next * self.vel_norm / v_norm
        loc[0, :, :], vel[0, :, :] = self._clamp(loc_next, vel_next)

        # disables division by zero warning, since I fix it with fill_diagonal
        with np.errstate(divide='ignore'):
            # half step leapfrog
            l2_dist_power3 = np.power(
                self._l2(loc_next.transpose(), loc_next.transpose()), 3. / 2.)

            # size of forces up to a 1/|r| factor
            # since I later multiply by an unnormalized r vector
            forces_size = self.interaction_strength * edges / l2_dist_power3
            np.fill_diagonal(forces_size,
                             0)  # self forces are zero (fixes division by zero)
            assert (np.abs(forces_size[diag_mask]).min() > 1e-10)
            F = (forces_size.reshape(1, n, n) *
                 np.concatenate((
                     np.subtract.outer(loc_next[0, :],
                                       loc_next[0, :]).reshape(1, n, n),
                     np.subtract.outer(loc_next[1, :],
                                       loc_next[1, :]).reshape(1, n, n),
                     np.subtract.outer(loc_next[2, :],
                                       loc_next[2, :]).reshape(1, n, n)))).sum(axis=-1)
            F[F > self._max_F] = self._max_F
            F[F < -self._max_F] = -self._max_F

            vel_next += self._delta_T * F
            # run leapfrog
            for i in range(1, T):
                loc_next += self._delta_T * vel_next
                #loc_next, vel_next = self._clamp(loc_next, vel_next)

                if i % sample_freq == 0:
                    loc[counter, :, :], vel[counter, :, :] = loc_next, vel_next
                    counter += 1

                l2_dist_power3 = np.power(
                    self._l2(loc_next.transpose(), loc_next.transpose()),
                    3. / 2.)
                forces_size = self.interaction_strength * edges / l2_dist_power3
                np.fill_diagonal(forces_size, 0)
                # assert (np.abs(forces_size[diag_mask]).min() > 1e-10)

                F = (forces_size.reshape(1, n, n) *
                     np.concatenate((
                         np.subtract.outer(loc_next[0, :],
                                           loc_next[0, :]).reshape(1, n, n),
                         np.subtract.outer(loc_next[1, :],
                                           loc_next[1, :]).reshape(1, n, n),
                         np.subtract.outer(loc_next[2, :],
                                           loc_next[2, :]).reshape(1, n, n)
                     ))).sum(axis=-1)
                F[F > self._max_F] = self._max_F
                F[F < -self._max_F] = -self._max_F
                vel_next += self._delta_T * F
            return loc, vel, edges, charges

In [3]:
sim = ChargedParticlesSim(n_balls=2, loc_std=2)
charges = np.array([[1.0], [-1.0]])
init_loc = np.array([[1,2,3],[1.1,2.1,3.1]])
init_vel = np.array([[1,1,0],[1,1,0]])
loc, vel, edges, charges = sim.sample_trajectory(charges = charges,init_loc = init_loc,init_vel=init_vel,T=5000, sample_freq=100)
t = time.time()
print(edges)
print("Simulation time: {}".format(time.time() - t))
vel_norm = np.sqrt((vel ** 2).sum(axis=1))
plt.figure()
axes = plt.gca()
axes.set_xlim([-10., 10.])
axes.set_ylim([-10., 10.])
for i in range(loc.shape[-1]):
    plt.plot(loc[:, 0, i], loc[:, 1, i])
    plt.plot(loc[0, 0, i], loc[0, 1, i], 'd')
plt.figure()
energies = [sim._energy(loc[i, :, :], vel[i, :, :], edges) for i in
            range(loc.shape[0])]
plt.plot(energies)
plt.show()

1.4736125994561546


/tmp/ipykernel_3187584/1845119746.py:89: RuntimeWarning: invalid value encountered in divide
  vel_next = vel_next * self.vel_norm / v_norm


ValueError: could not broadcast input array from shape (2,3) into shape (3,2)

In [19]:
loc[10]

array([[-2.10743827,  3.59512474,  1.88069564,  1.29508368, -0.5592499 ],
       [-2.42790635, -1.24755786,  2.16801485,  4.23430487,  3.57416654],
       [-1.17411927, -2.33794858,  0.45480186, -3.79945043,  0.70786731]])

In [17]:
vel[1]

array([[-0.05673636, -0.08749605, -0.48967021,  0.1210302 , -0.13157321],
       [-0.46228972, -0.47306893, -0.0104137 ,  0.4223156 ,  0.46087236],
       [-0.17962358, -0.09675919,  0.14665139, -0.23820734,  0.07417091]])

In [31]:
edges

array([[ 1.,  1., -1., -1., -1.],
       [ 1.,  1., -1., -1., -1.],
       [-1., -1.,  1.,  1.,  1.],
       [-1., -1.,  1.,  1.,  1.],
       [-1., -1.,  1.,  1.,  1.]])

In [79]:
charges

array([[ 1.],
       [-1.],
       [ 1.],
       [ 1.],
       [-1.]])

In [25]:
import numpy as np

# Define the initial conditions
charges = np.array([[1.0], [-1.0], [1.0], [1.0], [-1.0]])
init_loc = np.array([[-2.0541685,  3.70011251,  2.39385756,  1.1764005, -0.46754055],
                     [-1.96512326, -0.79247986,  2.17660365,  3.81140676,  3.13319916],
                     [-0.99574068, -2.24924995,  0.29601191, -3.56067901,  0.65457643]])
init_vel = np.array([[-0.05754886, -0.08298076, -0.48454586,  0.12170732, -0.14107747],
                     [-0.46237647, -0.47810277, -0.00952431,  0.422318,  0.46510115],
                     [-0.18013777, -0.09900656,  0.14378205, -0.23806592,  0.07966039]])

# Create the simulation instance
sim = ChargedParticlesSim(n_balls=5, loc_std=2)

# Run the simulation for 1 time step
loc, vel, edges, charges = sim.sample_trajectory(charges=charges, init_loc=init_loc, init_vel=init_vel, T=2, sample_freq=1)

# The predicted location in the next state
next_loc = loc[0]
next_loc


2.0
Force:[[ 0.00809043 -0.04537937 -0.0501968  -0.00687411  0.09435985]
 [ 0.00119847  0.05113968 -0.01064579 -0.0002812  -0.04141116]
 [ 0.00544586  0.02284303  0.02832656 -0.00134105 -0.05527439]]


/tmp/ipykernel_889532/4098411386.py:125: RuntimeWarning: invalid value encountered in power
  l2_dist_power3 = np.power(


array([[-2.05422609,  3.70002869,  2.39337825,  1.17652224, -0.46768368],
       [-1.96558605, -0.7929625 ,  2.17659422,  3.81182921,  3.13367129],
       [-0.99592097, -2.24934988,  0.29615415, -3.56091715,  0.65465725]])

In [26]:
loc.shape

(1, 3, 5)